# Seminar 05: CNNs From First Principles on FashionMNIST

**Student Version**

Goals for today:
- Reason about image tensors shaped **[B, C, H, W]**
- Track how convolution, activation, and pooling change feature maps
- Build a CNN from a verbal architecture description
- Compare CNN and MLP behavior without repeating training-loop boilerplate
- Inspect mistakes and feature maps to understand what the CNN is doing


## 0. Setup


In [ ]:
import math
import random
import time
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


### Configuration


In [ ]:
@dataclass
class Config:
    seed: int = 42
    train_n: object = 12000
    val_n: int = 2000
    batch_size: int = 128
    lr: float = 1e-3
    epochs: int = 5


cfg = Config()
cfg


### Helpers


In [ ]:
CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


def show_images(images, labels=None, preds=None, class_names=None, nrow=8, figsize=(12, 4)):
    images = images.detach().cpu()
    n_images = images.shape[0]
    ncol = int(math.ceil(n_images / nrow))
    plt.figure(figsize=figsize)
    for i in range(n_images):
        plt.subplot(ncol, nrow, i + 1)
        if images.shape[1] == 1:
            plt.imshow(images[i, 0], cmap='gray')
        else:
            plt.imshow(images[i].permute(1, 2, 0).numpy())
        plt.axis('off')
        title = None
        if labels is not None:
            y = int(labels[i])
            title = class_names[y] if class_names is not None else str(y)
        if preds is not None:
            p = int(preds[i])
            pred_txt = class_names[p] if class_names is not None else str(p)
            title = f'{title}\n-> {pred_txt}' if title is not None else pred_txt
        if title is not None:
            plt.title(title, fontsize=8)
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def accuracy_multiclass(logits, y_true):
    preds = logits.argmax(dim=1)
    return (preds == y_true).float().mean().item()


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def empty_history():
    return {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}


def plot_history(history, title_prefix=''):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure()
    plt.plot(epochs, history['train_loss'], label='train')
    plt.plot(epochs, history['val_loss'], label='val')
    plt.title(f'{title_prefix}Loss')
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.legend()
    plt.show()

    plt.figure()
    plt.plot(epochs, history['train_acc'], label='train')
    plt.plot(epochs, history['val_acc'], label='val')
    plt.title(f'{title_prefix}Accuracy')
    plt.xlabel('epoch')
    plt.ylabel('accuracy')
    plt.legend()
    plt.show()


def summarize_run(name, model, history, elapsed_sec):
    return {
        'name': name,
        'params': count_parameters(model),
        'best_val_acc': float(max(history['val_acc'])),
        'final_val_acc': float(history['val_acc'][-1]),
        'final_train_acc': float(history['train_acc'][-1]),
        'elapsed_sec': float(elapsed_sec),
    }


## 1. Provided data setup

FashionMNIST loading is infrastructure today. The seminar starts from inspecting the image tensor convention.


In [ ]:
def build_fashionmnist_loaders(cfg):
    transform = transforms.Compose([transforms.ToTensor()])
    train_full = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)

    generator = torch.Generator().manual_seed(cfg.seed)
    perm = torch.randperm(len(train_full), generator=generator)
    val_idx = perm[:cfg.val_n]
    train_idx = perm[cfg.val_n:]
    if cfg.train_n is not None:
        train_idx = train_idx[:cfg.train_n]

    train_ds = Subset(train_full, train_idx.tolist())
    val_ds = Subset(train_full, val_idx.tolist())

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, train_idx, val_idx


train_loader, val_loader, train_idx, val_idx = build_fashionmnist_loaders(cfg)
xb, yb = next(iter(train_loader))

print('xb:', xb.shape, xb.dtype, 'min/max:', float(xb.min()), float(xb.max()))
print('yb:', yb.shape, yb.dtype)
show_images(xb[:16], labels=yb[:16], class_names=CLASS_NAMES, nrow=8, figsize=(12, 4))

assert xb.ndim == 4 and xb.shape[1:] == (1, 28, 28)
assert yb.ndim == 1 and yb.dtype in (torch.int64, torch.long)
assert len(set(train_idx.tolist()).intersection(set(val_idx.tolist()))) == 0


## 2. Exercise 1: Convolution Shape Lab

Create a fake batch from this description:

> A batch of 4 grayscale images. Each image is 28 pixels high and 28 pixels wide.

Use PyTorch image tensor convention. Then pass it through two CNN blocks:
- first block: convolution creates 8 feature maps, then ReLU, then 2x2 max-pooling
- second block: convolution creates 16 feature maps, then ReLU, then 2x2 max-pooling

Basic `Conv2d` contract:
```python
nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)
```

Use `kernel_size=3` and padding that preserves height/width before pooling.

Fill the fake batch, layers, intermediate tensors, and `flatten_dim`. The expected `shape_trace` is already written, so use it as the target while you debug.

At the end, inspect the learnable parameter count for each layer and the total parameter count of this tiny convolution stack.


In [ ]:
# Exercise 1

x_fake = None
conv1 = None
relu = nn.ReLU()
pool = None
conv2 = None

h_conv1 = None
h_relu1 = None
h_pool1 = None
h_conv2 = None
h_relu2 = None
h_pool2 = None

shape_trace = {
    'input': (4, 1, 28, 28),
    'conv1': (4, 8, 28, 28),
    'relu1': (4, 8, 28, 28),
    'pool1': (4, 8, 14, 14),
    'conv2': (4, 16, 14, 14),
    'relu2': (4, 16, 14, 14),
    'pool2': (4, 16, 7, 7),
}

actual_shape_trace = {
    'input': tuple(x_fake.shape),
    'conv1': tuple(h_conv1.shape),
    'relu1': tuple(h_relu1.shape),
    'pool1': tuple(h_pool1.shape),
    'conv2': tuple(h_conv2.shape),
    'relu2': tuple(h_relu2.shape),
    'pool2': tuple(h_pool2.shape),
}

flatten_dim = None

param_counts = {
    'conv1': count_parameters(conv1),
    'relu': count_parameters(relu),
    'pool': count_parameters(pool),
    'conv2': count_parameters(conv2),
}
param_counts['total'] = sum(param_counts.values())

print('expected shapes:', shape_trace)
print('actual shapes  :', actual_shape_trace)
print('flatten_dim:', flatten_dim)
print('parameter counts:', param_counts)

assert actual_shape_trace == shape_trace
assert flatten_dim == 16 * 7 * 7
assert param_counts == {
    'conv1': 80,
    'relu': 0,
    'pool': 0,
    'conv2': 1168,
    'total': 1248,
}


## 3. Exercise 2: Build the CNN

Implement `SimpleCNN` from a verbal architecture description.

Architecture:
- input is a grayscale FashionMNIST image batch
- first convolution block produces 8 feature maps
- second convolution block receives those 8 maps and produces 16 feature maps
- both convolutions use 3x3 kernels and preserve spatial size before pooling
- each block uses ReLU and 2x2 max-pooling
- the classifier receives the flattened tensor after two pooling operations and outputs logits for 10 classes

Basic layer contracts:
```python
nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=1)
nn.MaxPool2d(kernel_size)
nn.Linear(in_features, out_features)
```

Contract for your model:
- `model(x)` returns logits shaped `[B, 10]`
- `model(x, return_features=True)` returns `(logits, features)`
- `features` contains `conv1`, `pool1`, `conv2`, `pool2`


In [ ]:
# Exercise 2

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = None
        self.relu1 = None
        self.pool1 = None
        self.conv2 = None
        self.relu2 = None
        self.pool2 = None
        self.classifier = None

    def forward(self, x, return_features=False):
        features = {}
        logits = None
        if return_features:
            return logits, features
        return logits


cnn = SimpleCNN().to(device)
with torch.no_grad():
    logits, features = cnn(xb[:8].to(device), return_features=True)

feature_shapes = {name: tuple(value.shape) for name, value in features.items()}
print('logits:', logits.shape)
print('features:', feature_shapes)
print('parameters:', count_parameters(cnn))

assert logits.shape == (8, 10)
assert feature_shapes == {
    'conv1': (8, 8, 28, 28),
    'pool1': (8, 8, 14, 14),
    'conv2': (8, 16, 14, 14),
    'pool2': (8, 16, 7, 7),
}
assert count_parameters(cnn) < 20000


## 4. CNN vs MLP


In [ ]:
loss_fn = nn.CrossEntropyLoss()


def train_one_epoch(model, loader, opt, loss_fn):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()

        bs = xb.shape[0]
        total_loss += loss.item() * bs
        total_acc += accuracy_multiclass(logits.detach(), yb) * bs
        total_n += bs

    return total_loss / total_n, total_acc / total_n


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        loss = loss_fn(logits, yb)

        bs = xb.shape[0]
        total_loss += loss.item() * bs
        total_acc += accuracy_multiclass(logits, yb) * bs
        total_n += bs

    return total_loss / total_n, total_acc / total_n


def run_training(name, model, cfg, train_loader, val_loader):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    history = empty_history()
    started = time.time()

    for epoch in range(1, cfg.epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, opt, loss_fn)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        print(f'{name:10s} epoch {epoch:02d} | train {tr_loss:.4f}/{tr_acc:.3f} | val {va_loss:.4f}/{va_acc:.3f}')

    elapsed_sec = time.time() - started
    return {'name': name, 'model': model, 'history': history, 'elapsed_sec': elapsed_sec}


class SimpleMLP(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.net(x)


seed_everything(cfg.seed)
cnn_result = run_training('cnn', SimpleCNN(), cfg, train_loader, val_loader)
plot_history(cnn_result['history'], title_prefix='CNN: ')
cnn_summary = summarize_run('cnn', cnn_result['model'], cnn_result['history'], cnn_result['elapsed_sec'])

seed_everything(cfg.seed)
mlp_result = run_training('mlp', SimpleMLP(), cfg, train_loader, val_loader)
plot_history(mlp_result['history'], title_prefix='MLP: ')
mlp_summary = summarize_run('mlp', mlp_result['model'], mlp_result['history'], mlp_result['elapsed_sec'])

comparison_rows = [cnn_summary, mlp_summary]
for row in comparison_rows:
    print(row)

print('CNN has fewer parameters:', cnn_summary['params'] < mlp_summary['params'])
print('CNN best val acc:', cnn_summary['best_val_acc'])
print('MLP best val acc:', mlp_summary['best_val_acc'])


## 5. Exercise 3: Collect Predictions

Implement `collect_predictions(model, loader)` for the trained CNN.

Contract:
- output dict keys: `images`, `y_true`, `y_pred`, `probs`
- `images` shape `[N, 1, 28, 28]`
- `y_true` and `y_pred` shape `[N]`
- `probs` shape `[N, 10]`

Useful hint: `torch.softmax(logits, dim=1)` converts logits to class probabilities.


In [ ]:
# Exercise 3

@torch.no_grad()
def collect_predictions(model, loader):
    output = None
    return output


preds = collect_predictions(cnn_result['model'], val_loader)

assert set(preds.keys()) == {'images', 'y_true', 'y_pred', 'probs'}
assert preds['images'].ndim == 4 and preds['images'].shape[1:] == (1, 28, 28)
assert preds['y_true'].shape == preds['y_pred'].shape
assert preds['probs'].shape == (preds['y_true'].shape[0], 10)
assert torch.allclose(preds['probs'].sum(dim=1), torch.ones(preds['probs'].shape[0]), atol=1e-5)
assert preds['images'].shape[0] == len(val_loader.dataset)


## 6. Exercise 4: Mistake Gallery and Per-Class Accuracy

Use the collected predictions to inspect model behavior.

Fill:
- `mistake_idx`, `mistake_images`, `mistake_true`, `mistake_pred`
- `per_class_correct`, `per_class_total`, `per_class_accuracy`
- `hardest_class_idx`

Useful hints:
- `mask.nonzero(as_tuple=False).reshape(-1)` gives indices where a boolean mask is true
- `torch.bincount(labels, minlength=10)` is useful for class counts
- `per_class_total.clamp_min(1)` replaces zeros by ones, so division cannot crash for a class with no examples
- visualize mistakes with `show_images(images, labels=true, preds=pred, class_names=CLASS_NAMES)`


Tip: avoid division-by-zero when computing per-class accuracy by clamping class counts to at least 1.

```python
# compute totals and avoid zeros
per_class_total = torch.bincount(y_true, minlength=10)
per_class_correct = torch.bincount(y_true[correct_mask], minlength=10)



In [ ]:
# Exercise 4

wrong_mask = None
mistake_idx = None
mistake_images = None
mistake_true = None
mistake_pred = None

per_class_correct = None
per_class_total = None
per_class_accuracy = None
hardest_class_idx = None

# Visualize up to 24 mistakes and print per-class accuracy.

assert mistake_images.shape[0] <= 24
assert mistake_images.ndim == 4 and mistake_images.shape[1:] == (1, 28, 28)
assert mistake_true.shape == mistake_pred.shape
assert per_class_correct.shape == (10,)
assert per_class_total.shape == (10,)
assert per_class_accuracy.shape == (10,)
assert 0 <= hardest_class_idx < 10

print('Hardest class:', hardest_class_idx, CLASS_NAMES[hardest_class_idx], float(per_class_accuracy[hardest_class_idx]))


## 7. Exercise 5: Feature Maps

Visualize what the trained CNN produces inside the network.

Fill:
- `first_layer_weights`: learned filters shaped `[8, 1, 3, 3]`
- plots of all first-layer filters
- `conv1_maps`: 8 early feature maps for one validation image
- `conv2_maps`: 16 deeper feature maps for the same image

Useful observations:
- early feature maps are often easier to visually interpret
- deeper feature maps can look less pretty but be more useful for classification


In [ ]:
# Exercise 5

trained_cnn = cnn_result['model']
first_layer_weights = None
one_image = None
one_image_logits = None
one_image_features = None
conv1_maps = None
conv2_maps = None

# Plot filters, conv1 feature maps, and conv2 feature maps.

assert first_layer_weights.shape == (8, 1, 3, 3)
assert one_image.shape == (1, 1, 28, 28)
assert one_image_logits.shape == (1, 10)
assert conv1_maps.shape == (1, 8, 28, 28)
assert conv2_maps.shape == (1, 16, 14, 14)


## 8. Optional discussion: CIFAR-10

What changes if we move from FashionMNIST to CIFAR-10?
- input channels: `1 -> 3`
- image size: `28x28 -> 32x32`
- after two pools: `32 -> 16 -> 8`, so flatten size becomes `16 * 8 * 8`
- the same tiny CNN is usually not strong enough


## 9. Wrap-up questions
1. Why is a CNN usually better suited to images than an MLP?
2. Why does padding matter before pooling?
3. What is the flatten size in this CNN and why?
4. Did the CNN use fewer parameters than the MLP?
5. Did fewer parameters automatically mean better validation accuracy?
6. Which classes were hardest, and do the mistakes look visually reasonable?
7. How do `conv1` and `conv2` feature maps differ visually?
